### We'll evaluate all 973 golden test questions against both persisted Chroma indexes.

Our objective is to answer one important question:

Does structure-based chunking retrieve government scheme information more effectively than recursive character chunking?

We'll keep the embedding model, query preprocessing, similarity configuration and Top-K identical.
Evaluation design
Configuration	Value
Test questions	973
EXP_001	Recursive Character
EXP_002_v2	Structure-Based
Embedding model	all-MiniLM-L6-v2
Maximum sequence length	256
Embedding batch size	16
Retrieval Top-K	20
Evaluation K	5, 10, 20

We'll calculate chunk-level precision and scheme-level Recall, Precision, MRR and NDCG. This is important because your current ground truth identifies relevant schemes, not exact relevant chunks.

In [1]:
import json
import math
import pandas as pd
import numpy as np
import chromadb
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# ============================================================
# EXP_003: DENSE RETRIEVAL EVALUATION
# EXP_001 vs EXP_002_v2
# ============================================================



# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

K_VALUES = [5, 10, 20]
MAX_K = max(K_VALUES)

QUERY_BATCH_SIZE = 32
EMBEDDING_BATCH_SIZE = 16

# Load the same embedding model and configuration
query_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

query_model.max_seq_length = 256

# Connect to existing Chroma indexes
client = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001"
)

collection_001 = client.get_collection("exp_001_recursive")

client_002 = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_002_v2"
)

collection_002 = client_002.get_collection("exp_002_structure_v2")

# Select one golden test question
golden_test_path = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits\\test.json")
with open(golden_test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)

# Keep query embedding configuration identical to indexing
query_model.max_seq_length = 256

experiments = {
    "EXP_001": collection_001,
    "EXP_002_v2": collection_002
}


# ------------------------------------------------------------
# 2. PREPARE GOLDEN TEST QUESTIONS
# ------------------------------------------------------------

valid_tests = [
    item for item in test_data
    if item.get("question")
    and item.get("relevant_scheme_ids")
]

questions = [item["question"] for item in valid_tests]

print("Total test questions:", len(questions))
print("Skipped questions:", len(test_data) - len(questions))


# ------------------------------------------------------------
# 3. GENERATE QUERY EMBEDDINGS ONCE
# ------------------------------------------------------------

query_embeddings = query_model.encode(
    questions,
    batch_size=EMBEDDING_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Query embeddings shape:", query_embeddings.shape)


# ------------------------------------------------------------
# 4. METRIC FUNCTIONS
# ------------------------------------------------------------

def calculate_recall(retrieved_ids, relevant_ids):

    if not relevant_ids:
        return 0.0

    hits = len(set(retrieved_ids) & relevant_ids)

    return hits / len(relevant_ids)


def calculate_precision(retrieved_ids, relevant_ids, k):

    hits = sum(
        1 for sid in retrieved_ids[:k]
        if sid in relevant_ids
    )

    return hits / k


def calculate_mrr(retrieved_ids, relevant_ids, k):

    for rank, sid in enumerate(retrieved_ids[:k], 1):

        if sid in relevant_ids:
            return 1.0 / rank

    return 0.0


def calculate_ndcg(retrieved_ids, relevant_ids, k):

    dcg = 0.0

    for rank, sid in enumerate(retrieved_ids[:k], 1):

        relevance = 1 if sid in relevant_ids else 0

        dcg += relevance / math.log2(rank + 1)

    ideal_relevant = min(len(relevant_ids), k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    return dcg / idcg if idcg else 0.0


# ------------------------------------------------------------
# 5. RETRIEVAL + EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_collection(collection, experiment_name):

    results_per_question = []

    total_questions = len(valid_tests)

    print(f"\nEvaluating {experiment_name}")

    for start in tqdm(
        range(0, total_questions, QUERY_BATCH_SIZE),
        desc=experiment_name
    ):

        end = min(start + QUERY_BATCH_SIZE, total_questions)

        batch_embeddings = query_embeddings[start:end]

        # Retrieve top 20 chunks for each query
        retrieved = collection.query(
            query_embeddings=batch_embeddings.tolist(),
            n_results=MAX_K,
            include=["metadatas", "distances"]
        )

        for local_idx, result_idx in enumerate(range(start, end)):

            test = valid_tests[result_idx]

            relevant_ids = set(
                test["relevant_scheme_ids"]
            )

            metadata_list = retrieved["metadatas"][local_idx]

            # Extract scheme IDs in ranking order
            chunk_scheme_ids = [
                metadata.get("scheme_id")
                for metadata in metadata_list
                if metadata and metadata.get("scheme_id")
            ]

            # Remove duplicate schemes while preserving rank
            unique_scheme_ids = list(
                dict.fromkeys(chunk_scheme_ids)
            )

            row = {
                "experiment": experiment_name,
                "test_id": test["test_id"],
                "question": test["question"],
                "scheme_id": test["scheme_id"],
                "relevant_scheme_ids": list(relevant_ids)
            }

            for k in K_VALUES:

                # Chunk-level precision
                row[f"chunk_precision@{k}"] = (
                    calculate_precision(
                        chunk_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                # Scheme-level metrics using unique schemes
                row[f"scheme_recall@{k}"] = (
                    calculate_recall(
                        unique_scheme_ids[:k],
                        relevant_ids
                    )
                )

                row[f"scheme_precision@{k}"] = (
                    calculate_precision(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                row[f"scheme_mrr@{k}"] = (
                    calculate_mrr(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

                row[f"scheme_ndcg@{k}"] = (
                    calculate_ndcg(
                        unique_scheme_ids,
                        relevant_ids,
                        k
                    )
                )

            results_per_question.append(row)

    return pd.DataFrame(results_per_question)


# ------------------------------------------------------------
# 6. RUN BOTH EXPERIMENTS
# ------------------------------------------------------------

df_001_results = evaluate_collection(
    collection_001,
    "EXP_001"
)

df_002_results = evaluate_collection(
    collection_002,
    "EXP_002_v2"
)


# ------------------------------------------------------------
# 7. COMBINE RESULTS
# ------------------------------------------------------------

all_results = pd.concat(
    [df_001_results, df_002_results],
    ignore_index=True
)


# ------------------------------------------------------------
# 8. GENERATE SUMMARY METRICS
# ------------------------------------------------------------

metric_columns = [
    column
    for column in all_results.columns
    if any(
        column.startswith(prefix)
        for prefix in [
            "chunk_precision",
            "scheme_recall",
            "scheme_precision",
            "scheme_mrr",
            "scheme_ndcg"
        ]
    )
]

summary = (
    all_results
    .groupby("experiment")[metric_columns]
    .mean()
    .reset_index()
)


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

output_dir = Path(
    "experiments/results/EXP_003_dense_retrieval"
)

output_dir.mkdir(parents=True, exist_ok=True)

all_results.to_csv(
    output_dir / "per_question_results.csv",
    index=False
)

summary.to_csv(
    output_dir / "summary.csv",
    index=False
)


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("DENSE RETRIEVAL EVALUATION COMPLETED")
print("=" * 65)

print("Evaluated questions:", len(valid_tests))

display(
    summary.round(4)
)

print("\nResults saved to:", output_dir)


g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 928.01it/s]


Total test questions: 973
Skipped questions: 0


Batches: 100%|██████████| 61/61 [00:32<00:00,  1.87it/s]


Query embeddings shape: (973, 384)

Evaluating EXP_001


EXP_001: 100%|██████████| 31/31 [00:17<00:00,  1.75it/s]



Evaluating EXP_002_v2


EXP_002_v2: 100%|██████████| 31/31 [00:16<00:00,  1.93it/s]



DENSE RETRIEVAL EVALUATION COMPLETED
Evaluated questions: 973


,experiment,chunk_precision@5,scheme_recall@5,scheme_precision@5,scheme_mrr@5,scheme_ndcg@5,chunk_precision@10,scheme_recall@10,scheme_precision@10,scheme_mrr@10,scheme_ndcg@10,chunk_precision@20,scheme_recall@20,scheme_precision@20,scheme_mrr@20,scheme_ndcg@20
0,EXP_001,0.2234,0.6886,0.1377,0.5933,0.6171,0.1496,0.7472,0.0747,0.6011,0.636,0.0960,0.7965,0.0398,0.6047,0.6487
1,EXP_002_v2,0.2181,0.6927,0.1385,0.5990,0.6224,0.1404,0.7533,0.0753,0.6071,0.642,0.0864,0.8078,0.0404,0.6110,0.6560



Results saved to: experiments\results\EXP_003_dense_retrieval


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 762.95it/s]


In [2]:

# EXP_002_v2: Structure-based chunking + MMR

from langchain_chroma import Chroma
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

# Connect to existing Chroma indexes
client = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001"
)

collection_001 = client.get_collection("exp_001_recursive")

vectorstore = Chroma(
    client=client,
    collection_name="exp_001_recursive",
    embedding_function=embedding_function
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 30,
        "lambda_mult": 0.5
    }
)

results = retriever.invoke(
    "I am a farmer and need a loan to buy fertilizer."
)

for rank, doc in enumerate(results, start=1):
    print(f"\nRank: {rank}")
    print("Scheme:", doc.metadata.get("scheme_id"))
    print("Section:", doc.metadata.get("section"))
    print("Content:", doc.page_content[:300])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 536.45it/s]



Rank: 1
Scheme: yhg4lZ0B5uF80_MGoGbk
Section: faq
Content: Question: Can this loan be used to purchase other farming equipment?
Answer: No, this loan is specifically intended for tractor purchases only.

Rank: 2
Scheme: 58czY54B0AGc0F0iERLw
Section: application_process
Content: Scheme: Scheme for Loan and Subsidy for Manure and Fertilizer-DNH&DD
Section: application_process

Rank: 3
Scheme: LqKpG54BXu-iUKXq61Sa
Section: faq
Content: Question: To be eligible to receive the financial assistance for promoting organic fertilizers, what specific online registration must I complete beforehand?
Answer: You must register your Bio-Gas or Compressed Bio-Gas units on the unified Galvanizing Organic Bio-Agro Resources Dhan portal managed b

Rank: 4
Scheme: waI1lZ0BXu-iUKXqsVA-
Section: faq
Content: Question: Who is eligible for this loan?
Answer: Farmers involved in sericulture within irrigated areas are eligible.

Rank: 5
Scheme: t8efwpwB0AGc0F0i6wUy
Section: faq
Content: Question: What kind of 

In [7]:
from pathlib import Path
import json
path = Path("G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\Raw\\all_schemes\\all_schemes.json")

with open(path, "r", encoding="utf-8") as f:
    all_schemes = json.load(f)

scheme_lookup = {
    scheme["scheme_id"]: scheme["name"]
    for scheme in all_schemes
}

In [8]:

# EXP_002_v2: Structure-based chunking + MMR

from langchain_chroma import Chroma
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 16
    }
)

# Connect to existing Chroma indexes
client = chromadb.PersistentClient(
    path="G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001"
)

collection_001 = client.get_collection("exp_001_recursive")

vectorstore = Chroma(
    client=client,
    collection_name="exp_001_recursive",
    embedding_function=embedding_function
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 30,
        "lambda_mult": 0.5
    }
)

results = retriever.invoke(
    "I am a farmer and need a loan to buy fertilizer."
)

for rank, doc in enumerate(results, start=1):

    scheme_id = doc.metadata.get("scheme_id")
    scheme_name = scheme_lookup.get(scheme_id, "Unknown Scheme")

    print(f"\nRank: {rank}")
    print("Scheme ID:", scheme_id)
    print("Scheme Name:", scheme_name)
    print("Section:", doc.metadata.get("section"))
    print("Content:", doc.page_content[:300])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.81it/s]



Rank: 1
Scheme ID: yhg4lZ0B5uF80_MGoGbk
Scheme Name: Primary Cooperative Agriculture and Rural Development Bank: For Tractor Purchase
Section: faq
Content: Question: Can this loan be used to purchase other farming equipment?
Answer: No, this loan is specifically intended for tractor purchases only.

Rank: 2
Scheme ID: 58czY54B0AGc0F0iERLw
Scheme Name: Scheme for Loan and Subsidy for Manure and Fertilizer-DNH&DD
Section: application_process
Content: Scheme: Scheme for Loan and Subsidy for Manure and Fertilizer-DNH&DD
Section: application_process

Rank: 3
Scheme ID: LqKpG54BXu-iUKXq61Sa
Scheme Name: Market Development Assistance
Section: faq
Content: Question: To be eligible to receive the financial assistance for promoting organic fertilizers, what specific online registration must I complete beforehand?
Answer: You must register your Bio-Gas or Compressed Bio-Gas units on the unified Galvanizing Organic Bio-Agro Resources Dhan portal managed b

Rank: 4
Scheme ID: waI1lZ0BXu-iUKXqsVA-


In [10]:
import pandas as pd
query = "I am a farmer and need a loan to buy fertilizer."

# Standard similarity retrieval
similarity_results = vectorstore.similarity_search(
    query,
    k=10
)

# MMR retrieval
mmr_results = vectorstore.max_marginal_relevance_search(
    query,
    k=10,
    fetch_k=30,
    lambda_mult=0.5
)

def get_scheme_ranking(docs):

    return [
        {
            "scheme_id": doc.metadata.get("scheme_id"),
            "scheme_name": scheme_lookup.get(
                doc.metadata.get("scheme_id")
            ),
            "section": doc.metadata.get("section"),
            "content": doc.page_content[:200]
        }
        for doc in docs
    ]

similarity_df = pd.DataFrame(
    get_scheme_ranking(similarity_results)
)

mmr_df = pd.DataFrame(
    get_scheme_ranking(mmr_results)
)

print("STANDARD SIMILARITY")
display(similarity_df)

print("MMR")
display(mmr_df)

STANDARD SIMILARITY


,scheme_id,scheme_name,section,content
0,yhg4lZ0B5uF80_MGoGbk,Primary Cooperative Agriculture and Rural Deve...,faq,Question: Can this loan be used to purchase ot...
1,58czY54B0AGc0F0iERLw,Scheme for Loan and Subsidy for Manure and Fer...,details,on a 40% subsidy and 60% loan basis. The assis...
2,t8efwpwB0AGc0F0i6wUy,Assistance to Farmers in Purchasing Nano Ferti...,documents_required,Scheme: Assistance to Farmers in Purchasing Na...
3,rRgplZ0B5uF80_MGRGaV,Primary Cooperative Agriculture and Rural Deve...,faq,Question: How can an applicant apply for the l...
4,58czY54B0AGc0F0iERLw,Scheme for Loan and Subsidy for Manure and Fer...,application_process,Scheme: Scheme for Loan and Subsidy for Manure...
5,58czY54B0AGc0F0iERLw,Scheme for Loan and Subsidy for Manure and Fer...,faq,Question: What type of assistance is provided ...
6,4Ma-F5gB0AGc0F0iYjUB,Primary Cooperative Agriculture and Rural Deve...,faq,Question: Who is eligible to apply for this lo...
7,LqKpG54BXu-iUKXq61Sa,Market Development Assistance,faq,Question: To be eligible to receive the financ...
8,waI1lZ0BXu-iUKXqsVA-,Primary Cooperative Agriculture and Rural Deve...,faq,Question: Who is eligible for this loan?\nAnsw...
9,jcboF5gB0AGc0F0i5Den,Primary Cooperative Agriculture and Rural Deve...,brief_description,Scheme: Primary Cooperative Agriculture and Ru...


MMR


,scheme_id,scheme_name,section,content
0,yhg4lZ0B5uF80_MGoGbk,Primary Cooperative Agriculture and Rural Deve...,faq,Question: Can this loan be used to purchase ot...
1,58czY54B0AGc0F0iERLw,Scheme for Loan and Subsidy for Manure and Fer...,application_process,Scheme: Scheme for Loan and Subsidy for Manure...
2,LqKpG54BXu-iUKXq61Sa,Market Development Assistance,faq,Question: To be eligible to receive the financ...
3,waI1lZ0BXu-iUKXqsVA-,Primary Cooperative Agriculture and Rural Deve...,faq,Question: Who is eligible for this loan?\nAnsw...
4,t8efwpwB0AGc0F0i6wUy,Assistance to Farmers in Purchasing Nano Ferti...,faq,Question: What kind of financial benefit does ...
5,LqKpG54BXu-iUKXq61Sa,Market Development Assistance,application_process,Fertilizer Management System portal.\n##### **...
6,LqKpG54BXu-iUKXq61Sa,Market Development Assistance,faq,Question: How must I execute the sales of my e...
7,2KFt_pgBXu-iUKXqs3ly,Distribution of Seeds of more productions vari...,faq,Question: What type of fertilizers are support...
8,cMa2F5gB0AGc0F0iITQ2,Chief Minister's Krishi Rinn Yojana,details,1. The farmer will get interest subvention of ...
9,lsbpF5gB0AGc0F0iYTfC,Primary Cooperative Agriculture and Rural Deve...,faq,Question: How can one apply for the loan?\nAns...


In [11]:
def retrieval_statistics(docs):
    scheme_ids = [
        doc.metadata.get("scheme_id")
        for doc in docs
    ]

    return {
        "total_chunks": len(docs),
        "unique_schemes": len(set(scheme_ids)),
        "duplicate_chunks": len(scheme_ids) - len(set(scheme_ids))
    }

stats = pd.DataFrame({
    "Similarity": retrieval_statistics(similarity_results),
    "MMR": retrieval_statistics(mmr_results)
}).T

display(stats)

,total_chunks,unique_schemes,duplicate_chunks
Similarity,10,8,2
MMR,10,8,2


In [12]:
similarity_ids = {
    doc.metadata.get("scheme_id")
    for doc in similarity_results
}

mmr_ids = {
    doc.metadata.get("scheme_id")
    for doc in mmr_results
}

print("Common schemes:", len(similarity_ids & mmr_ids))
print("Similarity-only:", len(similarity_ids - mmr_ids))
print("MMR-only:", len(mmr_ids - similarity_ids))

Common schemes: 5
Similarity-only: 3
MMR-only: 3


In [13]:
similarity_ids = [
    doc.metadata.get("scheme_id")
    for doc in similarity_results
]

mmr_ids = [
    doc.metadata.get("scheme_id")
    for doc in mmr_results
]

similarity_set = set(similarity_ids)
mmr_set = set(mmr_ids)

print("Common schemes:", len(similarity_set & mmr_set))
print("Similarity-only:", similarity_set - mmr_set)
print("MMR-only:", mmr_set - similarity_set)

print(
    "Scheme overlap:",
    round(
        len(similarity_set & mmr_set)
        / len(similarity_set | mmr_set) * 100,
        2
    ),
    "%"
)

Common schemes: 5
Similarity-only: {'4Ma-F5gB0AGc0F0iYjUB', 'rRgplZ0B5uF80_MGRGaV', 'jcboF5gB0AGc0F0i5Den'}
MMR-only: {'2KFt_pgBXu-iUKXqs3ly', 'lsbpF5gB0AGc0F0iYTfC', 'cMa2F5gB0AGc0F0iITQ2'}
Scheme overlap: 45.45 %
